**OBSOLETE -- superseded by the 2026-08-25 reorientation plan.**

Kept in the repo as a historical record (real, validated results at the
time -- e.g. the calibration submission that scored public LB 0.596) but
not maintained going forward. The gold+weak preprocessing/training/
inference pipeline built here is being rebuilt from scratch under the
new plan (measurement-gate fix, verified slice ordering, validated label
sets, a rebuilt `src/`-backed preprocessing pass, and a 6-slot
attention model), tracked in the `v2` notebooks
(`00v2_measurement_gate.ipynb`, `01v2_slice_ordering.ipynb`, ...). See
README.md for the current plan.

# 06 - Submission inference

This is the notebook actually **submitted** to the competition (Code
Competition rules: submit via Notebook, <=9h run-time, internet
disabled during scoring, `submission.csv` as output). It mounts the 5
checkpoints trained by `05c_gold_weak_checkpoint_train.ipynb`
(`gold_weak`/seed=42), preprocesses the real (hidden-until-scoring) test
set's DICOM live, and averages the 5 folds' predictions.

**Validated against `triplets_knee` (2026-08-24, `06b_preprocessing_validation.ipynb`):**
`05a_weak_dicom_preprocess.ipynb` -- the notebook that actually built
and validated the audit-fixed DICOM preprocessing pipeline on Kaggle --
was never synced back to this repo and is now confirmed permanently
gone from Kaggle too, so Cells 5-8 below (laterality resolution, the
laterality-based slice-order fix, VOI LUT intensity normalization) stay
a reconstruction from README prose, not a copy of the original code.
`06b` compared this reconstruction against all 58 gold studies'
already-known-good `triplets_knee` triplets:

- **Laterality slice-order direction: confirmed correct.** 58/58
  studies matched the reference in direct (not reversed) channel order,
  clean across L/R/unknown -- this was the one failure mode that could
  have silently corrupted every prediction, and it's ruled out.
- **Residual:** ~10/58 studies (spread across L/R/unknown, not
  laterality-correlated) show a moderate intensity mismatch (MAE up to
  ~0.24 on a [0,1] scale; median across all 58 is 0.0097). Three
  hypotheses were tested and ruled out (off-by-one center slice, gap
  size, `PhotometricInterpretation`/`RescaleSlope`/`RescaleIntercept`) --
  root cause not found, and can't be fully resolved without the
  original code. **Accepted as a known limitation for this calibration
  submission** (goal: one real leaderboard number to sanity-check the
  58-gold local CV, not a pixel-perfect reproduction) rather than
  chasing it further -- see
  `feedback_match_debugging_effort_to_stakes` memory. Revisit only if
  the real leaderboard score comes back implausibly bad.

Only trains/predicts with the `gold_weak`/seed=42 arm (5-fold ensemble)
-- this is a **calibration submission**, meant to answer "does our
58-gold local CV number mean anything on the real leaderboard", not the
best possible score.

## Cell 1 - Imports, mount test data + checkpoints

`CHECKPOINTS_MOUNT` is a placeholder -- fill in the real path after
publishing `05c`'s Dataset (same discovery-not-assumption pattern as
`05b` Cell 1: Kaggle datasets mount at `datasets/<owner>/<slug>/`, the
slug is Kaggle's own hyphenated version of the Dataset name).

In [ ]:
# Self-contained (no `from src import ...`) -- Kaggle doesn't mount this repo.
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"

# TODO: fill in after publishing 05c's checkpoint Dataset (verify by listing
# /kaggle/input/datasets/ rather than assuming the slug, same as 05b Cell 1).
CHECKPOINTS_MOUNT = Path("/kaggle/input/datasets/alherma7/folds-knee")
assert CHECKPOINTS_MOUNT.exists(), f"Checkpoints dataset not found at {CHECKPOINTS_MOUNT}"

with open(CHECKPOINTS_MOUNT / "metadata.json") as f:
    METADATA = json.load(f)
print("Checkpoint metadata:", METADATA)

FINDINGS = METADATA["findings"]
OFFICIAL_LABEL_COLUMNS = METADATA["official_label_columns"]
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())
CV_FOLDS = METADATA["cv_folds"]
BACKBONE = METADATA["backbone"]

test = pd.read_csv(RAW_DIR / "test.csv")
test_series = pd.read_csv(RAW_DIR / "test_series.csv")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")
print(f"test.csv: {len(test)} studies")
print(f"test_series.csv: {len(test_series)} series")
assert list(sample_submission.columns) == ["StudyInstanceUID"] + LABEL_COLS, \
    "sample_submission.csv column order does not match OFFICIAL_LABEL_COLUMNS"

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Cell 2 - Series selection

Ported **verbatim** from `04_baseline_cnn.ipynb` Cells 2/4
(`count_slices`/`select_sagittal_series`) -- this rule predates the
2026-08-18 audit and was never flagged as buggy (the audit's 4 P0 bugs
were laterality, metric leakage, intensity normalization, and the
augmentation bug downstream of intensity -- not series selection), so
it's ported with confidence rather than reconstructed. Adapted only to
take a `series_subdir` parameter (`"test_series"` instead of
`"train_series"`) and the `test_series.csv` frame.

In [ ]:
def count_slices(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    return len(list(d.glob("*.dcm")))


def select_sagittal_series(series_df, slice_counts):
    sagittal = series_df.copy()
    sagittal["n_slices"] = sagittal["SeriesInstanceUID"].map(slice_counts).fillna(0).astype(int)

    def _pick(group):
        fluid_sensitive = group[group["Fluid_Sensitive"] == 1]
        pool = fluid_sensitive if len(fluid_sensitive) > 0 else group
        pool = pool.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        return pool.iloc[0]

    return sagittal.groupby("StudyInstanceUID").apply(_pick, include_groups=False)


test_sag = test_series[test_series["Anatomical_Plane"].str.lower() == "sagittal"]
print(f"Sagittal series in test: {len(test_sag)} rows, covering {test_sag['StudyInstanceUID'].nunique()} / {len(test)} studies")

_slice_counts = {}
for _, row in test_sag.iterrows():
    key = row["SeriesInstanceUID"]
    _slice_counts[key] = count_slices(row["StudyInstanceUID"], key, "test_series")

selected = select_sagittal_series(test_sag, _slice_counts)
print(f"Series selected: {len(selected)} / {len(test)} studies")
missing_series = set(test["StudyInstanceUID"]) - set(selected.index)
print(f"Studies with no sagittal series at all: {len(missing_series)}")

## Cell 3 - Laterality resolution (RECONSTRUCTED, needs validation)

Reconstructed from README's description of `05a` Cell 5's finding
(2026-08-19): ~50% of studies have no usable DICOM `Laterality` tag;
~19% of those are an anonymization placeholder
(`SeriesDescription == "DummySeriesDesc!"`), the rest genuinely
unpopulated. Recovery: a `SeriesDescription` text-parsing fallback
(`L-`/`R-`/`LT.`/`RT.` prefixes, `LEFT`/`RIGHT` as a whole word), which
recovered 308/2,218 in `05a`'s run. The exact prefix/word-boundary
patterns below are a best-effort reconstruction of that description, not
copied from `05a`'s real (unsynced) code -- validate against a handful
of real DICOM before trusting this for the actual test set.

Studies where laterality stays unresolved are treated as
**"unknown", trained un-reversed** in `05c` (same convention as `05b`'s
training data) -- so unresolved test studies should be handled the same
way here, not dropped.

In [ ]:
import re

_LR_PREFIX_RE = re.compile(r"^\s*(L|R|LT|RT)[\s.\-_]", re.IGNORECASE)
_LEFT_WORD_RE = re.compile(r"\bLEFT\b", re.IGNORECASE)
_RIGHT_WORD_RE = re.compile(r"\bRIGHT\b", re.IGNORECASE)


def resolve_laterality(ds):
    """Return "L", "R", or "unknown" for one DICOM dataset (first slice of a series)."""
    tag_value = getattr(ds, "Laterality", None)
    if tag_value in ("L", "R"):
        return tag_value

    series_desc = getattr(ds, "SeriesDescription", None)
    if not isinstance(series_desc, str) or series_desc == "DummySeriesDesc!":
        return "unknown"

    prefix_match = _LR_PREFIX_RE.match(series_desc)
    if prefix_match:
        token = prefix_match.group(1).upper()
        return "L" if token in ("L", "LT") else "R"

    has_left = bool(_LEFT_WORD_RE.search(series_desc))
    has_right = bool(_RIGHT_WORD_RE.search(series_desc))
    if has_left and not has_right:
        return "L"
    if has_right and not has_left:
        return "R"
    return "unknown"

## Cell 4 - Slice loading with laterality-based order fix (RECONSTRUCTED, needs validation)

Reconstructed from the README's description of audit bug (1): sagittal
medial/lateral is the axis *along* the series, not an in-plane pixel
axis, so the fix is reversing the **slice list order**, not flipping
pixels. The measured fact from the audit: ascending `SliceLocation`
means lateral->medial for `Laterality == "R"` and medial->lateral for
`Laterality == "L"` -- opposite directions. To land on one consistent
order (medial->lateral) regardless of knee side, `R` studies need their
ascending-`SliceLocation` order reversed; `L` studies (and `unknown`,
matching `05c`'s training convention) keep ascending order as-is.

**This directionality (which side gets reversed) is inferred from the
README's prose, not verified against real pixel data in this session --
the single highest-priority cell to validate before a real
submission**, since getting it backwards would silently corrupt every
`R`-knee study's slice order.

In [ ]:
def load_series_slices_fixed(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    records = []
    first_ds = None
    for i, f in enumerate(files):
        ds = pydicom.dcmread(f)
        if i == 0:
            first_ds = ds
        records.append((float(ds.SliceLocation), ds.pixel_array, ds))
    records.sort(key=lambda r: r[0])

    laterality = resolve_laterality(first_ds)
    if laterality == "R":
        records = list(reversed(records))
    # "L" and "unknown" both keep ascending SliceLocation order (05c's training convention).

    slices = [pixels for _, pixels, _ in records]
    datasets = [ds for _, _, ds in records]
    return slices, datasets, laterality

## Cell 5 - Intensity normalization via VOI LUT (RECONSTRUCTED, needs validation)

Reconstructed from README's description of audit bug (3): raw `uint16`
DICOM pixels were fed straight into an ImageNet-pretrained backbone with
no rescaling. Fix: `pydicom.pixels.apply_voi_lut` (real DICOM VOI LUT,
`WindowCenter`/`WindowWidth`) with a percentile 0.5-99.5 fallback if
those tags are absent, then rescale to `[0, 1]`. The percentile bounds
and the min-max rescale step are inferred from the README's one-line
description, not copied from `05a`'s real code -- validate the output
range/visual appearance on a few real slices before trusting this.

In [ ]:
from pydicom.pixels import apply_voi_lut


def normalize_intensity(pixel_array, ds):
    has_window = hasattr(ds, "WindowCenter") and hasattr(ds, "WindowWidth")
    if has_window:
        try:
            windowed = apply_voi_lut(pixel_array, ds)
        except Exception:
            has_window = False
    if not has_window:
        lo, hi = np.percentile(pixel_array, [0.5, 99.5])
        windowed = np.clip(pixel_array, lo, hi)

    lo, hi = windowed.min(), windowed.max()
    if hi <= lo:
        return np.zeros_like(windowed, dtype=np.float32)
    return ((windowed - lo) / (hi - lo)).astype(np.float32)

## Cell 6 - Physical scale normalization + crop

Ported **verbatim** from `04_baseline_cnn.ipynb` Cell 14/16
(`normalize_physical_scale`/`center_crop_or_pad`) -- these were not part
of the audit's bug list either, only the pixel-flip half of the old
`normalize_laterality` (already replaced by Cell 4's slice-order fix
above, so the old function is intentionally not reused here).

In [ ]:
from scipy.ndimage import zoom


def normalize_physical_scale(pixel_array, pixel_spacing_mm, target_mm_per_pixel):
    factor = pixel_spacing_mm / target_mm_per_pixel
    return zoom(pixel_array, factor, order=1)


def center_crop_or_pad(pixel_array, crop_px):
    h, w = pixel_array.shape
    out = np.zeros((crop_px, crop_px), dtype=pixel_array.dtype)

    src_top = max(0, (h - crop_px) // 2)
    src_left = max(0, (w - crop_px) // 2)
    src = pixel_array[src_top:src_top + crop_px, src_left:src_left + crop_px]

    dst_top = max(0, (crop_px - h) // 2)
    dst_left = max(0, (crop_px - w) // 2)
    out[dst_top:dst_top + src.shape[0], dst_left:dst_left + src.shape[1]] = src
    return out


TARGET_MM_PER_PIXEL = 0.35
CROP_MM = 130.0
CROP_PX = round(CROP_MM / TARGET_MM_PER_PIXEL)
print(f"CROP_PX = {CROP_PX}")

## Cell 7 - 2.5D triplet construction

Ported **verbatim** from `04_baseline_cnn.ipynb` Cell 12
(`mm_to_slice_gap`/`sample_slice_indices`/`build_25d_triplet`) --
unaffected by the audit's bug list, and matches `GAP_MM=4.0` used in
both `05a`/`05b`.

In [ ]:
def mm_to_slice_gap(gap_mm, spacing_mm):
    return max(1, round(gap_mm / spacing_mm))


def sample_slice_indices(n_slices, n_triplets, gap):
    if n_slices <= 0 or n_triplets <= 0:
        raise ValueError("n_slices and n_triplets must be positive")
    lo, hi = gap, n_slices - 1 - gap
    if lo > hi:
        return [n_slices // 2] * n_triplets
    if n_triplets == 1:
        return [(lo + hi) // 2]
    return [int(round(x)) for x in np.linspace(lo, hi, num=n_triplets)]


def build_25d_triplet(slices, center_idx, gap):
    n = len(slices)
    idxs = [max(0, min(n - 1, center_idx + off)) for off in (-gap, 0, gap)]
    return np.stack([slices[i] for i in idxs], axis=0)


GAP_MM = 4.0

## Cell 8 - Per-study preprocessing function, with failure handling

Combines Cells 2-7 into one function per study, same try/except pattern
as `05a` Cell 9 (log the `StudyInstanceUID` + exception, don't abort the
whole run). A study that fails preprocessing gets no triplet; Cell 11
falls back to constant 0.5 for those rows rather than crashing the
submission.

In [ ]:
def slice_spacing_mm(study_id, series_id, series_subdir):
    """Ported verbatim from 04_baseline_cnn.ipynb Cell 8 -- priority:
    SpacingBetweenSlices, then SliceThickness, then the median absolute
    SliceLocation delta as a last resort (Fase 1: SpacingBetweenSlices
    absent in 14/165 series)."""
    d = RAW_DIR / series_subdir / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[0], stop_before_pixels=True)
    if "SpacingBetweenSlices" in ds0:
        return float(ds0.SpacingBetweenSlices)
    if "SliceThickness" in ds0:
        return float(ds0.SliceThickness)
    locs = sorted(float(pydicom.dcmread(f, stop_before_pixels=True).SliceLocation) for f in files)
    return float(np.median(np.abs(np.diff(locs))))


def preprocess_test_study(study_id, series_subdir="test_series"):
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    slices, datasets, laterality = load_series_slices_fixed(study_id, series_id, series_subdir)

    first_ds = datasets[0]
    pixel_spacing_mm = float(first_ds.PixelSpacing[0])
    spacing_between_slices_mm = slice_spacing_mm(study_id, series_id, series_subdir)

    gap = mm_to_slice_gap(GAP_MM, spacing_between_slices_mm)
    center = sample_slice_indices(len(slices), n_triplets=1, gap=gap)[0]
    idxs = [max(0, min(len(slices) - 1, center + off)) for off in (-gap, 0, gap)]

    normalized_channels = []
    for i in idxs:
        intensity_norm = normalize_intensity(slices[i], datasets[i])
        scaled = normalize_physical_scale(intensity_norm, pixel_spacing_mm, TARGET_MM_PER_PIXEL)
        cropped = center_crop_or_pad(scaled, CROP_PX)
        normalized_channels.append(cropped)

    triplet = np.stack(normalized_channels).astype(np.float32)
    return triplet, laterality


TRIPLETS_OUT = Path("/kaggle/working/test_triplets")
TRIPLETS_OUT.mkdir(parents=True, exist_ok=True)

failures = {}
laterality_counts = {"L": 0, "R": 0, "unknown": 0}
t0 = time.time()
for study_id in selected.index:
    try:
        triplet, laterality = preprocess_test_study(study_id)
        np.save(TRIPLETS_OUT / f"{study_id}.npy", triplet)
        laterality_counts[laterality] += 1
    except Exception as e:
        failures[study_id] = repr(e)

elapsed = time.time() - t0
n_done = len(selected) - len(failures)
print(f"Preprocessed: {n_done} / {len(selected)} in {elapsed:.1f}s ({elapsed / max(len(selected), 1):.3f}s/study)")
print(f"Failures: {len(failures)}")
for sid, err in list(failures.items())[:20]:
    print(f"  {sid[:25]}...: {err}")
print(f"Laterality breakdown: {laterality_counts}")

## Cell 9 - Sanity check on a sample triplet

Same spot-check `05a` Cell 11 did before publishing -- shape, dtype,
value range -- cheap enough to run before spending the full inference
budget on a broken pipeline.

In [ ]:
_sample_files = list(TRIPLETS_OUT.glob("*.npy"))
assert _sample_files, "No triplets were produced -- check the failures list in Cell 8 before continuing."
_sample = np.load(_sample_files[0])
print(f"Sample triplet: shape={_sample.shape}, dtype={_sample.dtype}, range=[{_sample.min():.4f}, {_sample.max():.4f}]")
assert _sample.shape == (3, CROP_PX, CROP_PX), f"Unexpected shape {_sample.shape}"
assert 0.0 <= _sample.min() and _sample.max() <= 1.0, "Intensity normalization did not land in [0, 1]"

## Cell 10 - Ensemble inference

Loads the 5 fold checkpoints from `05c`, builds the model architecture
with `pretrained=False` (no internet at submission time -- we load our
own fine-tuned weights, not ImageNet init, so this is not just allowed
but required), and averages the 5 folds' sigmoid probabilities per
study.

In [ ]:
class BaselineFindingModel(nn.Module):
    def __init__(self, backbone_name, n_findings, dropout=0.5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.num_features, n_findings),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


class TestTripletDataset(Dataset):
    def __init__(self, study_ids, triplets_dir):
        self.study_ids = list(study_ids)
        self.triplets_dir = triplets_dir

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        study_id = self.study_ids[idx]
        triplet = np.load(self.triplets_dir / f"{study_id}.npy")
        return torch.from_numpy(triplet), study_id


preprocessed_ids = [p.stem for p in TRIPLETS_OUT.glob("*.npy")]
test_ds = TestTripletDataset(preprocessed_ids, TRIPLETS_OUT)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

fold_predictions = []
for fold_idx in range(CV_FOLDS):
    checkpoint_path = CHECKPOINTS_MOUNT / f"fold{fold_idx}.pth"
    model = BaselineFindingModel(BACKBONE, n_findings=len(FINDINGS), pretrained=False).to(DEVICE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    preds, ids = [], []
    with torch.no_grad():
        for xb, batch_ids in test_loader:
            xb = xb.to(DEVICE)
            probs = torch.sigmoid(model(xb)).cpu().numpy()
            preds.append(probs)
            ids.extend(batch_ids)

    fold_df = pd.DataFrame(np.concatenate(preds), index=ids, columns=FINDINGS)
    fold_predictions.append(fold_df)
    print(f"fold {fold_idx}: predicted {len(fold_df)} studies")

    del model
    torch.cuda.empty_cache()

ensembled = sum(fold_predictions) / len(fold_predictions)
print(f"\nEnsembled predictions: {ensembled.shape}")

## Cell 11 - Build submission.csv

Every `StudyInstanceUID` in `test.csv` must appear exactly once (per the
Evaluation page's format) -- studies that failed preprocessing (Cell 8)
or have no ensembled prediction fall back to the constant 0.5 row
`sample_submission.csv` itself uses, rather than being dropped and
breaking the submission.

In [ ]:
submission = sample_submission.set_index("StudyInstanceUID").copy()
submission.loc[:, :] = 0.5  # default fallback for every row, overwritten below where we have a real prediction

_rename = {finding: OFFICIAL_LABEL_COLUMNS[finding] for finding in FINDINGS}
ensembled_official = ensembled.rename(columns=_rename)

_covered = submission.index.intersection(ensembled_official.index)
submission.loc[_covered, LABEL_COLS] = ensembled_official.loc[_covered, LABEL_COLS]

n_fallback = len(submission) - len(_covered)
print(f"Studies with a real prediction: {len(_covered)} / {len(submission)}")
print(f"Studies falling back to constant 0.5: {n_fallback}")

submission = submission.reset_index()
assert list(submission.columns) == ["StudyInstanceUID"] + LABEL_COLS
assert len(submission) == len(test), f"Expected {len(test)} rows, got {len(submission)}"
assert submission["StudyInstanceUID"].is_unique

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("\nWrote /kaggle/working/submission.csv")
print(submission.head())

## Closing task (after this notebook scores)

Report the leaderboard score against this conversation's calibration
question: does it land anywhere near the 0.5286 (gold-only) / 0.5711
(gold+weak) pooled OOF macro-AUC measured in `05b`/`05c`, or is it wildly
different -- which would mean the 58-gold local CV isn't a reliable
proxy for the real metric, and the architecture-lever conversation
(DINOv2, richer per-study input, LLM-derived weak labels) needs to
happen before trusting *any* further local-CV-only experiment. Before
that: if the score looks implausibly bad (e.g. near 0.5 exactly, or
NaN-like), re-check Cells 3-5 (the reconstructed-from-prose cells) first
-- that's the most likely place a silent bug would hide.